# Notebook Kaggle - RAM-H1200 Bone Segmentation
### Pipeline: DenseNet121 -> LayerCAM -> Bone Morphology -> SAM ViT-B -> Evaluation / U-Net

Notebook n?y theo c?u tr?c notebook c? nh?ng ?? chuy?n ho?n to?n sang **RAM-H1200**. M?i stage c? gi?i th?ch, input/output, sanity check, v? visualization m?u.

Output m?c ??nh l?u ? `/kaggle/working/ThesisOutputs`.

## 0. T?ng quan pipeline

```text
RAM-H1200 hand X-ray
  -> Stage 1: DenseNet121 hand checkpoint
  -> Stage 2: LayerCAM localization
  -> Stage 3: Bone morphology + connected components
  -> Stage 4: SAM box/point prompting
  -> Stage 5: Bone-aware mask selection + post-processing
  -> Pseudo mask
  -> Dice/IoU against RAM-H1200 GT

Optional supervised baseline:
RAM-H1200 image + GT mask -> U-Net -> final bone mask
```

## 1. C?u h?nh Kaggle

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
REPO_URL = 'https://github.com/itsthang333/Thesis.git'
BRANCH = 'TN_exp'
PROJECT_PARENT = WORKING / 'Thesis'
PROJECT_DIR = PROJECT_PARENT / 'project'
OUTPUT_ROOT = WORKING / 'ThesisOutputs'
SAM_CHECKPOINT = OUTPUT_ROOT / 'checkpoints' / 'sam_vit_b_01ec64.pth'

CLASSIFIER_OUTPUT = OUTPUT_ROOT / 'classifier'
PSEUDO_OUTPUT = OUTPUT_ROOT / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_ROOT / 'segmentation'
EVAL_CSV = OUTPUT_ROOT / 'ramh1200_eval.csv'
VIZ_OUTPUT = OUTPUT_ROOT / 'viz'
DEBUG_OUTPUT = OUTPUT_ROOT / 'debug_viz'
ABLATION_OUTPUT = OUTPUT_ROOT / 'ablation_preview'

IMAGE_SIZE = 384
NUM_WORKERS = 2
BATCH_SIZE_CLASSIFIER = 4
BATCH_SIZE_SEGMENTATION = 4
EPOCHS_CLASSIFIER = 25
EPOCHS_SEGMENTATION = 25

RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PSEUDO_MASKS = False   # False = preview 10 ?nh, True = ch?y full validation split
RUN_EVALUATE_PSEUDO = True
RUN_TRAIN_UNET = False
RUN_VISUALIZE_SAMPLE = True
RUN_DEBUG_SAM = True
RUN_ABLATION_PREVIEW = False

DATASET_OVERRIDE = ''  # N?u c?n, ??t path RAM-H1200 trong /kaggle/input t?i ??y.
HF_REPO_ID = 'TokyoTechMagicYang/RAM-H1200-v1'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Setup m?i tr??ng Kaggle

**M?c ti?u:** clone source nh?nh `TN_exp`, x? l? package c?n thi?t, v? tr?nh l?i PyTorch/CUDA tr?n GPU P100.

In [ ]:
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_PARENT)], check=True)
else:
    print('Project ?? t?n t?i:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())

import importlib.util
TORCH_ALREADY_IMPORTED = 'torch' in sys.modules

def run_pip(args, description):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f'C?i ??t th?t b?i: {description}')
    print(f'?? c?i: {description}')

try:
    gpu_query = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
    GPU_NAME = gpu_query.stdout.strip()
except Exception:
    GPU_NAME = ''
print(f'GPU: {GPU_NAME or "kh?ng ph?t hi?n"}')

# Tr?nh l?i Kaggle P100: b?n PyTorch m?c ??nh m?i c? th? kh?ng h? tr? sm_60.
if 'P100' in GPU_NAME:
    if TORCH_ALREADY_IMPORTED:
        raise RuntimeError('Torch ?? ???c import. H?y restart Kaggle session r?i ch?y l?i t? ??u.')
    print('C?i PyTorch 2.5.1 CUDA 11.8 h? tr? Pascal/sm_60...')
    run_pip(['--no-cache-dir', '--force-reinstall', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], 'PyTorch 2.5.1 CUDA 11.8')
    run_pip(['--no-cache-dir', '--force-reinstall', '--no-deps', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], 'torchvision 0.20.1')

run_pip(['pycocotools'], 'pycocotools')
run_pip(['opencv-python'], 'opencv-python')
run_pip(['--no-deps', 'git+https://github.com/facebookresearch/segment-anything.git'], 'segment-anything')

import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)

required_files = [
    PROJECT_DIR / 'datasets' / 'ramh1200.py',
    PROJECT_DIR / 'train_classifier.py',
    PROJECT_DIR / 'generate_pseudo_masks.py',
    PROJECT_DIR / 'evaluate_ramh1200_masks.py',
    PROJECT_DIR / 'train_segmentation.py',
    PROJECT_DIR / 'visualize_pipeline.py',
]
for path in required_files:
    print(('OK  ' if path.exists() else 'MISS'), path)

## 3. Resolve RAM-H1200 dataset tr?n Kaggle

?u ti?n dataset ?? attach qua **Add Input**. N?u kh?ng th?y v? Internet b?t, notebook t?i RAM-H1200 t? Hugging Face.

In [ ]:
def has_ram_layout(path: Path) -> bool:
    return (path / 'Segmentation').exists() or ((path / 'train').exists() and (path / 'train' / '_annotations_bone_rle.coco.json').exists())

def find_ram_root(base: Path):
    candidates = []
    if base and base.exists():
        candidates.append(base)
        candidates.extend(base.glob('*'))
        candidates.extend(base.glob('*/*'))
    for candidate in candidates:
        if (candidate / 'Segmentation').exists():
            return candidate
        if (candidate / 'train' / '_annotations_bone_rle.coco.json').exists():
            return candidate
    return None

print('Kaggle inputs ?? attach:')
if KAGGLE_INPUT.exists():
    for entry in sorted(KAGGLE_INPUT.iterdir()):
        print(' -', entry)

RAM_ROOT = find_ram_root(Path(DATASET_OVERRIDE)) if DATASET_OVERRIDE else find_ram_root(KAGGLE_INPUT)

if RAM_ROOT is None:
    print('Kh?ng t?m th?y RAM-H1200 trong /kaggle/input. Th? t?i t? Hugging Face...')
    run_pip(['huggingface_hub'], 'huggingface_hub')
    from huggingface_hub import snapshot_download
    RAM_ROOT = Path(snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        local_dir=str(WORKING / 'RAM-H1200-v1'),
        local_dir_use_symlinks=False,
    ))

print('RAM_ROOT:', RAM_ROOT)
if not has_ram_layout(RAM_ROOT):
    raise FileNotFoundError(f'Kh?ng th?y layout RAM-H1200 h?p l? t?i {RAM_ROOT}')

## 3. Ki?m tra dataset RAM-H1200

**M?c ti?u:** x?c nh?n notebook ?ang ??c ??ng RAM-H1200 v? annotation l? COCO RLE bone mask.

**Input:** `RAM_ROOT/Segmentation/<split>`.

**Output c?n th?y:** s? ?nh m?i split, category annotation, v? m?t ?nh m?u g?m ?nh g?c + GT mask + overlay.

In [ ]:
import json
from pathlib import Path

if not RAM_ROOT.exists():
    raise FileNotFoundError(f'Kh?ng t?m th?y RAM_ROOT: {RAM_ROOT}')

SEG_ROOT = RAM_ROOT / 'Segmentation' if (RAM_ROOT / 'Segmentation').exists() else RAM_ROOT
if not (SEG_ROOT / 'train').exists():
    raise FileNotFoundError(f'Kh?ng th?y split train trong {SEG_ROOT}')

print('SEG_ROOT:', SEG_ROOT)
for split in ['train', 'val', 'validation', 'test']:
    split_dir = SEG_ROOT / split
    if split_dir.exists():
        ann_path = split_dir / '_annotations_bone_rle.coco.json'
        images = sorted([p for p in split_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg', '.tif', '.tiff'}])
        print(f'{split:<10} images={len(images):>5} annotation={ann_path.exists()}')
        if ann_path.exists():
            data = json.loads(ann_path.read_text(encoding='utf-8'))
            cats = [c.get('name', c.get('id')) for c in data.get('categories', [])]
            print('  categories:', cats[:30])
            print('  coco images:', len(data.get('images', [])), 'annotations:', len(data.get('annotations', [])))

In [ ]:
from datasets.ramh1200 import RAMH1200SegmentationDataset
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

preview_ds = RAMH1200SegmentationDataset(root=RAM_ROOT, split='val', image_size=IMAGE_SIZE)
image_tensor, mask_tensor, image_name = preview_ds[0]
sample = preview_ds.samples[0]
image_path = Path(sample['image_path'])

original = Image.open(image_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
gt_mask = mask_tensor[0].numpy() > 0.5
original_np = np.array(original)
overlay = original_np.copy()
overlay[gt_mask] = (0.55 * overlay[gt_mask] + 0.45 * np.array([255, 40, 40])).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(original_np)
axes[0].set_title(f'?nh g?c\n{image_name}')
axes[1].imshow(gt_mask, cmap='gray')
axes[1].set_title('GT bone mask')
axes[2].imshow(overlay)
axes[2].set_title('Overlay GT l?n ?nh')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('image tensor:', tuple(image_tensor.shape), image_tensor.dtype)
print('mask tensor:', tuple(mask_tensor.shape), mask_tensor.dtype, 'mask pixels:', int(mask_tensor.sum().item()))

## 4. Chu?n b? SAM checkpoint

**M?c ti?u:** Stage 2 c?n SAM ViT-B ?? refine v?ng x??ng t? prompt.

**Input:** file `sam_vit_b_01ec64.pth`, n?u ch?a c? th? t?i t? ngu?n ch?nh th?c.

**Output:** path checkpoint t?n t?i tr??c khi generate pseudo mask.

In [ ]:
if not SAM_CHECKPOINT.exists():
    print('Ch?a th?y SAM checkpoint, ?ang t?i SAM ViT-B...')
    import urllib.request
    SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    urllib.request.urlretrieve(url, SAM_CHECKPOINT)
else:
    print('SAM checkpoint ?? t?n t?i:', SAM_CHECKPOINT)

## 5. Stage 1 - DenseNet121 hand checkpoint

**Nhi?m v? trong pipeline:** cung c?p feature map v? gradient cho LayerCAM.

**Input:** ?nh hand X-ray t? RAM-H1200.

**Output:** `best_classifier.pt` v? `training_log.csv`.

RAM-H1200 trong project n?y ch? d?ng nh?n `hand`, v? v?y checkpoint n?y kh?ng c?n l? b? ph?n lo?i nhi?u v?ng gi?i ph?u nh? tr??c.

In [ ]:
classifier_cmd = [
    sys.executable, 'train_classifier.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_CLASSIFIER),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_CLASSIFIER),
    '--output-dir', str(CLASSIFIER_OUTPUT),
]
print(' '.join(classifier_cmd))
if RUN_TRAIN_CLASSIFIER:
    subprocess.run(classifier_cmd, check=True)
else:
    print('RUN_TRAIN_CLASSIFIER=False, b? qua train classifier.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / 'best_classifier.pt'
log_path = CLASSIFIER_OUTPUT / 'training_log.csv'
print('Classifier checkpoint:', CLASSIFIER_CHECKPOINT, 'exists=', CLASSIFIER_CHECKPOINT.exists())
if log_path.exists():
    df = pd.read_csv(log_path)
    display(df.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[['train_loss', 'val_loss']].plot(ax=axes[0], title='Classifier loss')
    df[['train_f1', 'val_f1']].plot(ax=axes[1], title='Classifier F1')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ch?a c? training_log.csv cho classifier.')

## 6. Stage 2 - Sinh pseudo mask b?ng LayerCAM + morphology + SAM

**Nhi?m v? trong pipeline:** t?o mask x??ng gi? t? ?nh X-ray, DenseNet checkpoint v? SAM.

**Input:** ?nh RAM-H1200 validation split, classifier checkpoint, SAM checkpoint.

**C?c b??c b?n trong:**
1. DenseNet -> LayerCAM.
2. Bone morphology -> candidate components.
3. Component bbox + structured points -> SAM prompt.
4. Ch?n SAM mask b?ng `bone_hybrid`.
5. Post-process b?o to?n x??ng nh? v? khe gi?a c?c x??ng.

**Output:** `pseudo_masks/masks/*.png` v? `pseudo_masks/overlays/*_fused_layercam.png`.

In [ ]:
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(f'Thi?u classifier checkpoint: {CLASSIFIER_CHECKPOINT}. H?y ch?y Stage 1 tr??c.')

process_args = ['--process-all'] if RUN_FULL_PSEUDO_MASKS else ['--max-images', '10']
pseudo_cmd = [
    sys.executable, 'generate_pseudo_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--target-columns', 'hand',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', '1',
    '--num-workers', str(NUM_WORKERS),
    '--confidence-threshold', '0.5',
    '--cam-percentile', '85.0',
    '--max-points', '5',
    '--min-component-area', '100',
    '--mask-score-threshold', '0.4',
    '--selection-method', 'bone_hybrid',
    '--fusion-topk', '3',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--max-bone-components', '6',
    '--points-per-component', '3',
    '--bbox-padding-ratio', '0.05',
    '--negative-points-per-component', '0',
    '--bone-seed-percentile', '88',
    '--bone-support-percentile', '62',
    '--closing-kernel', '5',
    '--opening-kernel', '0',
    '--max-hole-area', '500',
    '--min-size', '40',
    '--save-visuals-limit', '10',
    *process_args,
    '--output-dir', str(PSEUDO_OUTPUT),
]
print(' '.join(pseudo_cmd))
subprocess.run(pseudo_cmd, check=True)

### 6.1 Hi?n th? sample Stage 2

Cell n?y gi?p xem tr?c quan: CAM c? b?m v?ng x??ng kh?ng, pseudo mask c? ph? ???c h?nh kh?i x??ng kh?ng, v? c? b? d?nh m? m?m qu? nhi?u kh?ng.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

mask_dir = PSEUDO_OUTPUT / 'masks'
overlay_dir = PSEUDO_OUTPUT / 'overlays'
mask_files = sorted(mask_dir.glob('*.png'))
print('S? pseudo masks:', len(mask_files), mask_dir)

n = min(5, len(mask_files))
if n:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1:
        axes = np.array([axes])
    val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
    for row, mask_path in zip(axes, mask_files[:n]):
        stem = mask_path.stem
        image_match = next(iter(sorted(val_dir.glob(f'{stem}.*'))), None)
        overlay_match = next(iter(sorted(overlay_dir.glob(f'{stem}*fused_layercam.png'))), None)
        if image_match:
            row[0].imshow(Image.open(image_match).convert('RGB'))
            row[0].set_title(f'?nh g?c\n{stem}')
        if overlay_match:
            row[1].imshow(Image.open(overlay_match).convert('RGB'))
            row[1].set_title('LayerCAM overlay')
        row[2].imshow(Image.open(mask_path), cmap='gray')
        row[2].set_title('Pseudo mask')
        for ax in row:
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Ch?a c? pseudo mask.')

## 7. ??nh gi? pseudo mask v?i GT RAM-H1200

**M?c ti?u:** ?o ch?t l??ng pseudo mask b?ng Dice/IoU so v?i GT bone mask.

N?u `RUN_FULL_PSEUDO_MASKS=False`, k?t qu? ch? ph?n ?nh 10 ?nh preview. Khi b?o c?o ch?nh th?c, h?y b?t `RUN_FULL_PSEUDO_MASKS=True` r?i ch?y l?i Stage 2 v? Stage 7.

In [ ]:
eval_cmd = [
    sys.executable, 'evaluate_ramh1200_masks.py',
    '--ram-root', str(RAM_ROOT),
    '--split', 'val',
    '--pred-mask-root', str(PSEUDO_OUTPUT / 'masks'),
    '--image-size', str(IMAGE_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--output-csv', str(EVAL_CSV),
]
print(' '.join(eval_cmd))
if RUN_EVALUATE_PSEUDO:
    subprocess.run(eval_cmd, check=True)
else:
    print('RUN_EVALUATE_PSEUDO=False, b? qua evaluate.')

In [ ]:
if EVAL_CSV.exists():
    eval_df = pd.read_csv(EVAL_CSV)
    display(eval_df.tail(10))
    valid = eval_df[eval_df['status'] == 'ok'].copy()
    if not valid.empty:
        print('S? ?nh evaluate:', len(valid))
        print('Mean Dice:', valid['dice'].mean())
        print('Mean IoU :', valid['iou'].mean())
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        valid['dice'].hist(ax=axes[0], bins=20)
        axes[0].set_title('Ph?n b? Dice pseudo mask')
        valid['iou'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Ph?n b? IoU pseudo mask')
        plt.tight_layout()
        plt.show()
else:
    print('Ch?a c? file evaluation CSV.')

## 8. Stage 3 - Supervised U-Net baseline

**Nhi?m v?:** train U-Net tr?c ti?p b?ng GT mask th?t c?a RAM-H1200 ?? c? baseline supervised.

**Input:** image + GT bone mask.

**Output:** `best_unet.pt`, training curves loss/Dice/IoU.

In [ ]:
unet_cmd = [
    sys.executable, 'train_segmentation.py',
    '--ram-root', str(RAM_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_SEGMENTATION),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_SEGMENTATION),
    '--output-dir', str(SEG_OUTPUT),
]
print(' '.join(unet_cmd))
if RUN_TRAIN_UNET:
    subprocess.run(unet_cmd, check=True)
else:
    print('RUN_TRAIN_UNET=False, b? qua train U-Net.')

In [ ]:
seg_log = SEG_OUTPUT / 'training_log.csv'
seg_ckpt = SEG_OUTPUT / 'best_unet.pt'
print('U-Net checkpoint:', seg_ckpt, 'exists=', seg_ckpt.exists())
if seg_log.exists():
    seg_df = pd.read_csv(seg_log)
    display(seg_df.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    seg_df[['train_loss', 'val_loss']].plot(ax=axes[0], title='U-Net loss')
    seg_df[['train_dice', 'val_dice']].plot(ax=axes[1], title='U-Net Dice')
    seg_df[['train_iou', 'val_iou']].plot(ax=axes[2], title='U-Net IoU')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ch?a c? training_log.csv cho U-Net.')

## 9. Visualize to?n b? pipeline tr?n m?t ?nh

Cell n?y t?o pipeline strip: ?nh g?c, LayerCAM, bone guidance, prompt, SAM candidates, pseudo mask, v? U-Net overlay n?u c? checkpoint U-Net.

In [ ]:
val_dir = SEG_ROOT / 'val' if (SEG_ROOT / 'val').exists() else SEG_ROOT / 'validation'
sample_image = next(iter(sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.bmp', '.png', '.jpg', '.jpeg'}])), None)
print('Sample image:', sample_image)

viz_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(sample_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--output-path', str(VIZ_OUTPUT / f'{sample_image.stem}_pipeline.png'),
]
if (SEG_OUTPUT / 'best_unet.pt').exists():
    viz_cmd.extend(['--segmentation-checkpoint', str(SEG_OUTPUT / 'best_unet.pt')])
print(' '.join(viz_cmd))
if RUN_VISUALIZE_SAMPLE and sample_image is not None:
    subprocess.run(viz_cmd, check=True)
else:
    print('RUN_VISUALIZE_SAMPLE=False ho?c kh?ng t?m th?y ?nh m?u.')

In [ ]:
viz_files = sorted(VIZ_OUTPUT.glob('*_pipeline.png'))
if viz_files:
    img = Image.open(viz_files[-1])
    plt.figure(figsize=(22, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(viz_files[-1].name)
    plt.show()
else:
    print('Ch?a c? pipeline strip.')

## 10. Debug SAM candidates

Gi?ng notebook c?, ph?n n?y m? r?ng debug: hi?n th? t?ng SAM candidate mask, overlay, v? `scores.json`. D?ng stage n?y khi pseudo mask b? l?ch, thi?u x??ng ho?c d?nh m? m?m qu? nhi?u.

In [ ]:
debug_image = sample_image
DEBUG_OUTPUT.mkdir(parents=True, exist_ok=True)
debug_strip = DEBUG_OUTPUT / f'{debug_image.stem}_debug_pipeline.png'

debug_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--image-path', str(debug_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--debug',
    '--output-path', str(debug_strip),
]
print(' '.join(debug_cmd))
if RUN_DEBUG_SAM:
    subprocess.run(debug_cmd, check=True)
else:
    print('RUN_DEBUG_SAM=False, b? qua debug SAM.')

In [ ]:
import json

if debug_strip.exists():
    plt.figure(figsize=(22, 4))
    plt.imshow(Image.open(debug_strip))
    plt.axis('off')
    plt.title('Debug pipeline strip')
    plt.show()

debug_dir = DEBUG_OUTPUT / 'debug' / debug_image.stem
mask_files = sorted(debug_dir.glob('mask_*.png'))
overlay_files = sorted(debug_dir.glob('overlay_mask_*.png'))
scores_path = debug_dir / 'scores.json'
print('debug_dir:', debug_dir)
print('S? SAM candidate masks:', len(mask_files))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])
    for i, mf in enumerate(mask_files):
        axes[0][i].imshow(Image.open(mf), cmap='gray')
        axes[0][i].set_title(mf.stem)
        axes[0][i].axis('off')
        if i < len(overlay_files):
            axes[1][i].imshow(Image.open(overlay_files[i]).convert('RGB'))
        axes[1][i].set_title('overlay')
        axes[1][i].axis('off')
    plt.tight_layout()
    plt.show()

if scores_path.exists():
    scores = json.loads(scores_path.read_text(encoding='utf-8'))
    print('scores.json')
    for key, value in scores.items():
        score = value.get('score', 0)
        area = value.get('area', 0)
        bar = '#' * int(score * 30)
        print(f'{key:<8} score={score:.3f} area={area:>8} {bar}')

## 11. Ablation preview nh?

Kh?ng b?t bu?c. B?t `RUN_ABLATION_PREVIEW=True` ?? so s?nh m?t s? prompt/scoring strategy tr?n c?ng m?t ?nh.

In [ ]:
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)
ABLATION_STRATEGIES = [
    ('box_point_bone_hybrid', ['--sam-prompt-mode', 'box_point', '--selection-method', 'bone_hybrid']),
    ('point_bone_hybrid', ['--sam-prompt-mode', 'point', '--selection-method', 'bone_hybrid']),
    ('box_point_coverage', ['--sam-prompt-mode', 'box_point', '--selection-method', 'coverage']),
]

if RUN_ABLATION_PREVIEW:
    for label, extra in ABLATION_STRATEGIES:
        out_path = ABLATION_OUTPUT / f'{label}_pipeline.png'
        cmd = [
            sys.executable, 'visualize_pipeline.py',
            '--image-path', str(debug_image),
            '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
            '--sam-checkpoint', str(SAM_CHECKPOINT),
            '--image-size', str(IMAGE_SIZE),
            '--morphology-fusion-mode', 'components',
            '--output-path', str(out_path),
            *extra,
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)
else:
    print('RUN_ABLATION_PREVIEW=False, b? qua ablation preview.')

In [ ]:
ablation_files = sorted(ABLATION_OUTPUT.glob('*_pipeline.png'))
if ablation_files:
    fig, axes = plt.subplots(len(ablation_files), 1, figsize=(22, 4*len(ablation_files)))
    if len(ablation_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, ablation_files):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Ch?a c? ablation output.')

## 12. Checklist khi b?o c?o k?t qu?

N?n l?u l?i c?c artifact sau:

1. ?nh g?c + GT mask RAM-H1200.
2. LayerCAM overlay + pseudo mask.
3. Pipeline strip t? `visualize_pipeline.py`.
4. B?ng Dice/IoU t? `evaluate_ramh1200_masks.py`.
5. Training curves c?a U-Net supervised baseline.
6. Ablation prompt/scoring n?u c? thay ??i tham s?.